# Dataset 3: prior utilization, payer, and medication EDA

This split contains emergency and inpatient utilization, payer code, medication changes, and drug-status variables.

The notebook analyses this split independently. The original CSV is not changed; the cleaned copy is saved in `EDA_Dataset` for later joining on `encounter_id`.

## Setup

In [1]:
from pathlib import Path  # Import required library

import matplotlib.pyplot as plt  # Import required library
import pandas as pd  # Import required library
import seaborn as sns  # Import required library

sns.set_theme(style="whitegrid", context="notebook")  # Set the chart theme
pd.set_option("display.max_rows", 100)  # Configure dataframe display
pd.set_option("display.max_columns", 50)  # Configure dataframe display

In [2]:
def show_category_counts(series, title, top_n=15):  # Define the categorical-frequency chart helper
    counts = series.fillna("Missing").value_counts().head(top_n).sort_values()  # Calculate category frequencies
    fig, ax = plt.subplots(figsize=(10, max(3.5, 0.38 * len(counts))))  # Create the chart area
    ax.barh(counts.index.astype(str), counts.values, color="#4C78A8")  # Draw the horizontal bar chart
    ax.set_title(f"{title}: top {top_n} categories")  # Label the chart axis or title
    ax.set_xlabel("Encounter count")  # Label the chart axis or title
    ax.set_ylabel("")  # Label the chart axis or title
    for y, value in enumerate(counts.values):  # Add value labels to each bar
        ax.text(value, y, f" {value:,}", va="center", fontsize=8)  # Run this EDA or data-cleaning step
    plt.tight_layout()  # Adjust chart spacing
    plt.show()  # Display the chart


def show_numeric_distribution(series, title, discrete=False):  # Define the numeric-distribution chart helper
    values = series.dropna()  # Remove missing numeric values before plotting
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))  # Create the chart area
    sns.histplot(values, discrete=discrete, ax=axes[0], color="#4C78A8")  # Plot the feature distribution
    axes[0].set_title(f"{title}: distribution")  # Configure the numeric chart
    axes[0].set_xlabel(title)  # Configure the numeric chart
    sns.boxplot(x=values, ax=axes[1], color="#72B7B2")  # Plot the feature box plot
    axes[1].set_title(f"{title}: box plot")  # Configure the numeric chart
    axes[1].set_xlabel(title)  # Configure the numeric chart
    plt.tight_layout()  # Adjust chart spacing
    plt.show()  # Display the chart

## Load the source data

In [3]:
source_path = Path("divided_datasets") / "dataset_3.csv"  # Set the path to the source CSV
df_raw = pd.read_csv(source_path, na_values=["?"], keep_default_na=True, low_memory=False)  # Load the source CSV and convert question marks to missing values

## Initial inspection

In [4]:
df_raw.head()  # Inspect the original dataframe

,encounter_id,glyburide-metformin,change,number_emergency,payer_code,number_inpatient,repaglinide,rosiglitazone,glyburide
0,2278392,No,No,0,NaN,0,No,No,No
1,149190,No,Ch,0,NaN,0,No,No,No
2,64410,No,No,0,NaN,1,No,No,No
3,500364,No,Ch,0,NaN,0,No,No,No
4,16680,No,Ch,0,NaN,0,No,No,No


In [5]:
df_raw.info()  # Inspect the original dataframe

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype
---  ------               --------------   -----
 0   encounter_id         101766 non-null  int64
 1   glyburide-metformin  101766 non-null  str  
 2   change               101766 non-null  str  
 3   number_emergency     101766 non-null  int64
 4   payer_code           61510 non-null   str  
 5   number_inpatient     101766 non-null  int64
 6   repaglinide          101766 non-null  str  
 7   rosiglitazone        101766 non-null  str  
 8   glyburide            101766 non-null  str  
dtypes: int64(3), str(6)
memory usage: 8.2 MB


In [6]:
df_raw.shape  # Inspect the original dataframe

(101766, 9)

In [7]:
df_raw.describe()  # Inspect the original dataframe

,encounter_id,number_emergency,number_inpatient
count,1.017660e+05,101766.000000,101766.000000
mean,1.652016e+08,0.197836,0.635566
std,1.026403e+08,0.930472,1.262863
min,1.252200e+04,0.000000,0.000000
25%,8.496119e+07,0.000000,0.000000
50%,1.523890e+08,0.000000,0.000000
75%,2.302709e+08,0.000000,1.000000
max,4.438672e+08,76.000000,21.000000


In [8]:
df_raw.isna().sum().sort_values(ascending=False).to_frame(name="missing_count")  # Inspect the original dataframe

,missing_count
payer_code,40256
encounter_id,0
glyburide-metformin,0
change,0
number_emergency,0
number_inpatient,0
repaglinide,0
rosiglitazone,0
glyburide,0


In [9]:
df_raw.duplicated().sum()  # Inspect the original dataframe

np.int64(0)

In [10]:
df_raw["encounter_id"].duplicated().sum()  # Run this EDA or data-cleaning step

np.int64(0)

In [11]:
df_raw["encounter_id"].is_unique  # Run this EDA or data-cleaning step

True

In [12]:
df_raw.nunique(dropna=False).sort_values().to_frame(name="unique_values_including_missing")  # Inspect the original dataframe

,unique_values_including_missing
change,2
glyburide-metformin,4
repaglinide,4
rosiglitazone,4
glyburide,4
payer_code,18
number_inpatient,21
number_emergency,33
encounter_id,101766


## Cleaning decisions

- Missing-value normalization: Question-mark and blank text values are treated as missing.
- Payer code: A missingness flag is retained and absent payer codes are labeled Unknown for visualization.
- Rows and join key: No rows are removed. encounter_id remains unchanged and unique for the later join.

In [13]:
df = df_raw.copy()  # Copy the source dataframe before cleaning

text_columns = df.select_dtypes(include=["object", "string"]).columns  # Identify text columns for normalization
for column in text_columns:  # Clean each text column
    df[column] = df[column].astype("string").str.strip().replace("", pd.NA)  # Strip whitespace and convert blank text to missing values

df["payer_code_missing"] = df["payer_code"].isna()  # Create or handle payer-code missingness information
df["payer_code"] = df["payer_code"].fillna("Unknown")  # Create or handle payer-code missingness information
columns_removed = []  # Record columns removed during cleaning

assert len(df) == len(df_raw), "No encounter rows should be removed"  # Validate the cleaning result
assert df["encounter_id"].notna().all(), "The join key must not be missing"  # Validate the cleaning result
assert df["encounter_id"].is_unique, "The join key must remain unique"  # Validate the cleaning result
assert df.columns.is_unique, "Duplicate column names are not allowed"  # Validate the cleaning result

In [14]:
cleaning_summary = pd.DataFrame({  # Create the cleaning-decision summary
    "decision": ["Missing-value normalization","Payer code","Rows and join key","Columns removed"],  # Add a cleaning-summary field
    "detail": ["Question-mark and blank text values are treated as missing.","A missingness flag is retained and absent payer codes are labeled Unknown for visualization.","No rows are removed. encounter_id remains unchanged and unique for the later join.","Columns removed after review: none"]  # Add a cleaning-summary field
})  # Run this EDA or data-cleaning step
cleaning_summary  # Create the cleaning-decision summary

,decision,detail
0,Missing-value normalization,Question-mark and blank text values are treate...
1,Payer code,A missingness flag is retained and absent paye...
2,Rows and join key,No rows are removed. encounter_id remains unch...
3,Columns removed,Columns removed after review: none


## Post-cleaning validation

In [15]:
df.head()  # Run this EDA or data-cleaning step

,encounter_id,glyburide-metformin,change,number_emergency,payer_code,number_inpatient,repaglinide,rosiglitazone,glyburide,payer_code_missing
0,2278392,No,No,0,Unknown,0,No,No,No,True
1,149190,No,Ch,0,Unknown,0,No,No,No,True
2,64410,No,No,0,Unknown,1,No,No,No,True
3,500364,No,Ch,0,Unknown,0,No,No,No,True
4,16680,No,Ch,0,Unknown,0,No,No,No,True


In [16]:
df.info()  # Run this EDA or data-cleaning step

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 10 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   encounter_id         101766 non-null  int64 
 1   glyburide-metformin  101766 non-null  string
 2   change               101766 non-null  string
 3   number_emergency     101766 non-null  int64 
 4   payer_code           101766 non-null  string
 5   number_inpatient     101766 non-null  int64 
 6   repaglinide          101766 non-null  string
 7   rosiglitazone        101766 non-null  string
 8   glyburide            101766 non-null  string
 9   payer_code_missing   101766 non-null  bool  
dtypes: bool(1), int64(3), string(6)
memory usage: 8.6 MB


In [17]:
df.shape  # Run this EDA or data-cleaning step

(101766, 10)

In [18]:
df.describe()  # Run this EDA or data-cleaning step

,encounter_id,number_emergency,number_inpatient
count,1.017660e+05,101766.000000,101766.000000
mean,1.652016e+08,0.197836,0.635566
std,1.026403e+08,0.930472,1.262863
min,1.252200e+04,0.000000,0.000000
25%,8.496119e+07,0.000000,0.000000
50%,1.523890e+08,0.000000,0.000000
75%,2.302709e+08,0.000000,1.000000
max,4.438672e+08,76.000000,21.000000


In [19]:
df.isna().sum().sort_values(ascending=False).to_frame(name="missing_count_after_cleaning")  # Run this EDA or data-cleaning step

,missing_count_after_cleaning
encounter_id,0
glyburide-metformin,0
change,0
number_emergency,0
payer_code,0
number_inpatient,0
repaglinide,0
rosiglitazone,0
glyburide,0
payer_code_missing,0


In [20]:
df["encounter_id"].nunique()  # Inspect the selected cleaned column

101766

## Univariate EDA after cleaning

### Prior emergency visits

In [21]:
df["number_emergency"].describe()  # Inspect the selected cleaned column

count    101766.000000
mean          0.197836
std           0.930472
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          76.000000
Name: number_emergency, dtype: float64

In [22]:
show_numeric_distribution(df["number_emergency"], "Prior emergency visits", discrete=True)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_17896\4155684497.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Prior inpatient visits

In [23]:
df["number_inpatient"].describe()  # Inspect the selected cleaned column

count    101766.000000
mean          0.635566
std           1.262863
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max          21.000000
Name: number_inpatient, dtype: float64

In [24]:
show_numeric_distribution(df["number_inpatient"], "Prior inpatient visits", discrete=True)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_17896\4155684497.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Glyburide-metformin status

In [25]:
df["glyburide-metformin"].value_counts(dropna=False).head(10).to_frame(name="count")  # Inspect the selected cleaned column

,count
glyburide-metformin,
No,101060
Steady,692
Up,8
Down,6


In [26]:
show_category_counts(df["glyburide-metformin"], "Glyburide-metformin status", top_n=10)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_17896\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Medication change

In [27]:
df["change"].value_counts(dropna=False).head(10).to_frame(name="count")  # Inspect the selected cleaned column

,count
change,
No,54755
Ch,47011


In [28]:
show_category_counts(df["change"], "Medication change", top_n=10)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_17896\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Payer code

In [29]:
df["payer_code"].value_counts(dropna=False).head(20).to_frame(name="count")  # Create or handle payer-code missingness information

,count
payer_code,
Unknown,40256
MC,32439
HM,6274
SP,5007
BC,4655
MD,3532
CP,2533
UN,2448
CM,1937


In [30]:
show_category_counts(df["payer_code"], "Payer code", top_n=20)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_17896\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Repaglinide status

In [31]:
df["repaglinide"].value_counts(dropna=False).head(10).to_frame(name="count")  # Inspect the selected cleaned column

,count
repaglinide,
No,100227
Steady,1384
Up,110
Down,45


In [32]:
show_category_counts(df["repaglinide"], "Repaglinide status", top_n=10)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_17896\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Rosiglitazone status

In [33]:
df["rosiglitazone"].value_counts(dropna=False).head(10).to_frame(name="count")  # Inspect the selected cleaned column

,count
rosiglitazone,
No,95401
Steady,6100
Up,178
Down,87


In [34]:
show_category_counts(df["rosiglitazone"], "Rosiglitazone status", top_n=10)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_17896\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Glyburide status

In [35]:
df["glyburide"].value_counts(dropna=False).head(10).to_frame(name="count")  # Inspect the selected cleaned column

,count
glyburide,
No,91116
Steady,9274
Up,812
Down,564


In [36]:
show_category_counts(df["glyburide"], "Glyburide status", top_n=10)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_17896\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


## Numeric relationships

In [37]:
numeric_columns = ["number_emergency", "number_inpatient"]  # Select numeric columns for relationship analysis
df[numeric_columns].corr()  # Inspect the selected cleaned column

,number_emergency,number_inpatient
number_emergency,1.000000,0.266559
number_inpatient,0.266559,1.000000


In [38]:
correlation = df[numeric_columns].corr()  # Calculate the numeric correlation matrix
plt.figure(figsize=(5, 4))  # Create the chart area
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="Blues", vmin=-1, vmax=1)  # Plot the numeric correlation heatmap
plt.title("Correlation between prior-use measures")  # Set the chart title
plt.tight_layout()  # Adjust chart spacing
plt.show()  # Display the chart

C:\Users\anind\AppData\Local\Temp\ipykernel_17896\3507867548.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


## Save the cleaned split

In [39]:
output_dir = Path("EDA_Dataset")  # Set the cleaned-data output folder
output_dir.mkdir(exist_ok=True)  # Set the cleaned-data output folder
output_path = output_dir / "dataset_3.csv"  # Set the cleaned CSV output path
df.to_csv(output_path, index=False)  # Save the cleaned dataframe
print(f"Saved {df.shape[0]:,} rows and {df.shape[1]:,} columns to {output_path}")  # Report the saved dataset details

Saved 101,766 rows and 10 columns to EDA_Dataset\dataset_3.csv


In [40]:
saved_df = pd.read_csv(output_path)  # Reload the saved CSV for verification
assert saved_df.shape == df.shape, "Saved CSV shape does not match the cleaned dataframe"  # Validate the cleaning result
saved_df.head()  # Reload the saved CSV for verification

,encounter_id,glyburide-metformin,change,number_emergency,payer_code,number_inpatient,repaglinide,rosiglitazone,glyburide,payer_code_missing
0,2278392,No,No,0,Unknown,0,No,No,No,True
1,149190,No,Ch,0,Unknown,0,No,No,No,True
2,64410,No,No,0,Unknown,1,No,No,No,True
3,500364,No,Ch,0,Unknown,0,No,No,No,True
4,16680,No,Ch,0,Unknown,0,No,No,No,True
